In [ ]:
from import_libraries import *  # NOSONAR # NOSONAR
from collections import Counter
import subprocess  # nosec B404

In [ ]:
max_complexity = 11
def parse_cyclomatic_complexity(file_path: str) -> tuple[int, float, list[str]]:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract average complexity
    avg_match = re.search(r"Average complexity: [A-Z] \(([\d.]+)\)", content)
    avg_score = float(avg_match.group(1)) if avg_match else 0.0

    # Extract number of analyzed blocks
    total_match = re.search(r"(\d+)\s+blocks \(classes, functions, methods\) analyzed\.", content)
    total_blocks = int(total_match.group(1)) if total_match else None

    flagged = []
    for line in content.splitlines():
        if " - D (" in line or " - E (" in line or " - C (" in line:
            flagged.append(line.strip())

    return total_blocks, avg_score, flagged


def parse_maintainability(file_path: str) -> tuple[list[str], float]:
    grade_weights = {"A": 3, "B": 2, "C": 1}
    grade_counts = Counter()
    total_files = 0
    flagged = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            match = re.search(r"- ([A-Z]) \([\d.]+\)", line)
            if match:
                grade = match.group(1)
                grade_counts[grade] += 1
                total_files += 1
                if grade != "A":
                    flagged.append(line.strip())

    if total_files == 0:
        return [], 0.0, 0.0

    # Calculate weighted index (normalized to 0–100)
    weighted_index = (
        sum(grade_weights.get(grade, 0) * count for grade, count in grade_counts.items())
        / (3 * total_files)
    ) * 100

    grade_distribution = {
        grade: round((count / total_files) * 100, 2) for grade, count in grade_counts.items()
    }

    return flagged, round(weighted_index, 2), grade_distribution


def classify_comment_density(density: float) -> str:
    """Classify comment density into qualitative labels.

    Parameters
    ----------
    density : float
        Comment density as a percentage of LOC.

    Returns
    -------
    str
        Qualitative label.
    """
    if density >= 50:
        return "🏆 Excellent"
    elif 35 <= density < 50:
        return "✅ Very Good"
    elif 25 <= density < 35:
        return "👍 Good"
    elif 15 <= density < 25:
        return "⚠️ Fair"
    else:
        return "🚨 Poor"


def normalized_comment_score(comment_avg: float) -> float:
    """Convert comment density to a normalized score based on tiers."""
    if comment_avg >= 50:
        # Map 50–100 → 90–100
        return 90 + (comment_avg - 50) * (10 / 50)
    elif 35 <= comment_avg < 50:
        return 75 + (comment_avg - 35) * (15 / 15)
    elif 25 <= comment_avg < 35:
        return 60 + (comment_avg - 25) * (15 / 10)
    elif 15 <= comment_avg < 25:
        return 40 + (comment_avg - 15) * (20 / 10)
    else:
        return comment_avg * (39 / 15)  # Map 0–15 → 0–39


def parse_comment_density(file_path: str) -> tuple[list[str], float, str]:
    densities = []
    flagged = []
    current_file = None
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip().endswith(".py"):
                current_file = line.strip()
            match = re.search(r"\(C \+ M % L\): (\d+)%", line)
            if match:
                pct = int(match.group(1))
                densities.append(pct)
                if pct < 25 and current_file:
                    flagged.append(f"{current_file}: {pct}%")
    avg_density = sum(densities) / len(densities) if densities else 0.0
    density_label = classify_comment_density(avg_density)
    return flagged, avg_density, density_label


def calculate_code_quality_score(complexity: float, mi_avg: float, comment_avg: float) -> float:
    complexity_score = max(0.0, 100 - complexity * 10)
    comment_score = normalized_comment_score(comment_avg)
    return round((mi_avg + comment_score + complexity_score) / 3, 2)


def generate_code_quality_report(
    path_complexity: str,
    path_maintainability: str,
    path_raw_metrics: str
) -> None:
    print("📊 Generating Code Quality Report...\n")

    # Parse all files
    total_blocks, avg_complexity, complex_funcs = parse_cyclomatic_complexity(path_complexity)
    mi_flags, mi_avg, mi_distribution = parse_maintainability(path_maintainability)
    low_doc_files, comment_avg, comment_label = parse_comment_density(path_raw_metrics)

    # Calculate score
    cqs = calculate_code_quality_score(avg_complexity, mi_avg, comment_avg)
    if not total_blocks:
        raise RuntimeError("Error: there aren't any functions checked.")

    # Print section-wise summary
    print("✅ 1. Cyclomatic Complexity (from cyclomatic_complexity.txt)")
    print(f"Total functions/methods analyzed: {total_blocks}")
    print(f"Average complexity score: A ({avg_complexity:.2f}) → {'Excellent' if avg_complexity <= max_complexity else 'Needs Refactor'}\n")
    if complex_funcs:
        print("Functions with C or worse complexity:")
        for line in complex_funcs[:10]:
            print(f"  - {line}")
        if len(complex_funcs) > max_complexity:
            print("  ...")

    print("\n✅ 2. Maintainability Index (from maintainability_index.txt)")
    print(f"Average maintainability index: {mi_avg:.2f}")
    print(f"Maintainability distribution: {mi_distribution}")

    if mi_flags:
        print("Modules with maintainability issues:")
        for line in mi_flags:
            print(f"  - {line}")
    else:
        print("All files are well-maintained (A ratings).")

    print("\n✅ 3. Raw Metrics and Documentation (from raw_metrics.txt)")
    print(f"Average comment density: {comment_avg:.2f}% → {comment_label}")
    if low_doc_files:
        print("Low-commented files (<25%):")
        for line in low_doc_files:
            print(f"  - {line}")
    else:
        print("All files have acceptable documentation levels.")

    print("\n🎯 Final Code Quality Score: ", f"{cqs} / 100")


In [ ]:
def run_quality_checks(batch_path: str) -> None:
    """Run the Radon batch script and ensure it completes successfully."""
    batch_file = Path(batch_path)
    if not batch_file.is_file():
        raise FileNotFoundError(f"Batch file not found: {batch_file}")

    print(f"🔧 Running: {batch_file} ...")
    result = subprocess.run(
        [str(batch_file)],
        check=True, shell=False,
        cwd=batch_file.parent,  # Ensures paths in BAT are resolved correctly
        capture_output=True,
        text=True,
        timeout=3600,
    )

    if result.returncode != 0:
        print("❌ Error running batch file:\n", result.stderr)
        raise RuntimeError("Batch execution failed.")

    print("✅ Batch file executed successfully.\n")
    print(result.stdout)

In [ ]:
py_path = os.getenv("PYTHONPATH")
if py_path:
    bat_path = os.path.split(py_path)[0]
else:
    # Fallback to current working directory if PYTHONPATH is not set
    bat_path = os.getcwd()

print(bat_path)
txt_path = os.path.join(bat_path, "src", "notebooks", "quality_reports")
file_cyclomatic_complexity = os.path.join(txt_path, "cyclomatic_complexity.txt")
file_maintainability_index = os.path.join(txt_path, "maintainability_index.txt")
file_raw_metrics = os.path.join(txt_path, "raw_metrics.txt")

run_quality_checks(os.path.join(bat_path, "run_quality_checks.bat"))

generate_code_quality_report(
    file_cyclomatic_complexity,
    file_maintainability_index,
    file_raw_metrics
)

In [ ]:
run_quality_checks(os.path.join(bat_path, "run_code_checks.bat"))